# How do vision models fail, and can we see why?
**ViT vs CNN robustness and explainability on traffic signs (GTSRB)**

This notebook fine-tunes a small Vision Transformer (DeiT-Tiny) and a CNN (ResNet-18), then:
1. measures accuracy on clean images and on **simulated** fog, blur, low light and noise at 3 severities;
2. visualises *where each model looks* (Grad-CAM for the CNN, attention rollout for the ViT);
3. runs a simple **deletion faithfulness test** (does removing the highlighted region hurt the prediction more than removing a random region?);
4. measures parameters and inference latency.

**Before you run:** Runtime > Change runtime type > GPU (T4). Then Runtime > Run all (about 15-25 minutes).

**Honest scope:** small data subset, one random seed, simplified corruption simulations, latency measured on a Colab GPU (not edge hardware). Treat results as an exploratory study, not a benchmark.

In [ ]:
!pip -q install timm

## 1. Setup

In [ ]:
import os, time, random, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import datasets, transforms
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms.functional as TF
from torch.utils.data import DataLoader, Subset
import timm
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU only: switch to a GPU runtime, training will be very slow")
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| timm", timm.__version__)

CFG = dict(
    n_train=6000,        # training images (random subset of GTSRB train)
    n_test=2000,         # test images (random subset of GTSRB test)
    img_size=224,
    batch_size=64,
    epochs=3,
    lr=1e-4,
    severities=[1, 2, 3],
    out_dir="results",
)
os.makedirs(CFG["out_dir"], exist_ok=True)
NUM_CLASSES = 43

## 2. Data (GTSRB subset)
Images stay in [0, 1] here. Normalisation happens inside the model wrapper, so corruptions are applied to raw pixel values.

In [ ]:
tfm = transforms.Compose([
    transforms.Resize((CFG["img_size"], CFG["img_size"])),
    transforms.ToTensor(),
])
train_full = datasets.GTSRB(root="data", split="train", download=True, transform=tfm)
test_full  = datasets.GTSRB(root="data", split="test",  download=True, transform=tfm)

rng = np.random.RandomState(SEED)
train_idx = rng.choice(len(train_full), CFG["n_train"], replace=False)
test_idx  = rng.choice(len(test_full),  CFG["n_test"],  replace=False)

train_loader = DataLoader(Subset(train_full, train_idx), batch_size=CFG["batch_size"],
                          shuffle=True, num_workers=2, pin_memory=(DEVICE == "cuda"))
test_loader  = DataLoader(Subset(test_full, test_idx), batch_size=CFG["batch_size"],
                          shuffle=False, num_workers=2, pin_memory=(DEVICE == "cuda"))
print("train images:", len(train_idx), "| test images:", len(test_idx))

## 3. Simulated corruptions
Simple, transparent approximations (not physically exact):
- **fog**: blend the image with a light-grey haze
- **blur**: Gaussian blur
- **low_light**: darken with a gamma curve and scale
- **noise**: additive Gaussian noise

In [ ]:
def fog(x, s):
    a = [0.2, 0.4, 0.6][s - 1]
    haze = torch.full_like(x, 0.85)
    return (x * (1 - a) + haze * a).clamp(0, 1)

def blur(x, s):
    sigma = [1.0, 2.0, 3.5][s - 1]
    k = int(2 * round(3 * sigma) + 1)          # odd kernel size
    return TF.gaussian_blur(x, kernel_size=[k, k], sigma=[sigma, sigma])

def low_light(x, s):
    scale = [0.6, 0.4, 0.2][s - 1]
    return ((x.clamp(0, 1) ** 1.5) * scale).clamp(0, 1)

def noise(x, s):
    std = [0.05, 0.10, 0.20][s - 1]
    return (x + torch.randn_like(x) * std).clamp(0, 1)

CORRUPTIONS = {"fog": fog, "blur": blur, "low_light": low_light, "noise": noise}

# Sanity check: look at the corruptions before trusting any numbers
x_demo, _ = next(iter(test_loader))
x_demo = x_demo[:1].to(DEVICE)
panels = [("clean", x_demo)] + [(f"{c} (sev 3)", fn(x_demo, 3)) for c, fn in CORRUPTIONS.items()]
fig, axes = plt.subplots(1, len(panels), figsize=(3 * len(panels), 3.2))
for ax, (t, img) in zip(axes, panels):
    ax.imshow(img[0].permute(1, 2, 0).cpu().numpy()); ax.set_title(t); ax.axis("off")
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/corruption_examples.png", dpi=150); plt.show()

## 4. Models

In [ ]:
class Wrapped(nn.Module):
    """Takes images in [0,1], applies ImageNet normalisation, then the model."""
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std",  torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.m((x - self.mean) / self.std)

def build_resnet():
    m = resnet18(weights=ResNet18_Weights.DEFAULT)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return Wrapped(m)

def build_vit():
    m = timm.create_model("deit_tiny_patch16_224", pretrained=True, num_classes=NUM_CLASSES)
    return Wrapped(m)

models = {"ResNet-18 (CNN)": build_resnet(), "DeiT-Tiny (ViT)": build_vit()}
for n, m in models.items():
    print(n, "| params (M):", round(sum(p.numel() for p in m.parameters()) / 1e6, 2))

## 5. Fine-tune both models (same recipe for a fair comparison)

In [ ]:
def train_model(model, name):
    model.to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=0.01)
    scaler = torch.cuda.amp.GradScaler(enabled=(DEVICE == "cuda"))
    log = []
    for ep in range(CFG["epochs"]):
        model.train(); t0 = time.time(); correct = 0; seen = 0; loss_sum = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=(DEVICE == "cuda")):
                out = model(x)
                loss = F.cross_entropy(out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); seen += len(y)
        row = dict(model=name, epoch=ep + 1, loss=loss_sum / seen, train_acc=correct / seen, seconds=time.time() - t0)
        log.append(row); print(row)
    return log

train_logs = []
for name, m in models.items():
    print("Training", name)
    train_logs += train_model(m, name)
pd.DataFrame(train_logs).to_csv(f"{CFG['out_dir']}/training_log.csv", index=False)

## 6. Accuracy under corruptions

In [ ]:
@torch.no_grad()
def accuracy(model, corruption_fn=None, severity=0):
    model.eval(); correct = 0; seen = 0
    for x, y in test_loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        if corruption_fn is not None:
            x = corruption_fn(x, severity)
        correct += (model(x).argmax(1) == y).sum().item(); seen += len(y)
    return correct / seen

rows = []
for name, m in models.items():
    set_seed(); rows.append(dict(model=name, corruption="clean", severity=0, accuracy=accuracy(m)))
    for cname, fn in CORRUPTIONS.items():
        for s in CFG["severities"]:
            set_seed()   # same noise pattern for every model
            rows.append(dict(model=name, corruption=cname, severity=s, accuracy=accuracy(m, fn, s)))
df = pd.DataFrame(rows)
df.to_csv(f"{CFG['out_dir']}/accuracy_results.csv", index=False)
print(df.pivot_table(index=["corruption", "severity"], columns="model", values="accuracy").round(3))

In [ ]:
corrs = list(CORRUPTIONS.keys())
clean = df[df["corruption"] == "clean"].set_index("model")["accuracy"]
fig, axes = plt.subplots(1, len(corrs), figsize=(4.2 * len(corrs), 3.6), sharey=True)
for ax, c in zip(axes, corrs):
    for name in models:
        ys = [clean[name]] + df[(df["model"] == name) & (df["corruption"] == c)].sort_values("severity")["accuracy"].tolist()
        ax.plot([0] + CFG["severities"], ys, marker="o", label=name)
    ax.set_title(c); ax.set_xlabel("severity"); ax.set_xticks([0] + CFG["severities"]); ax.grid(alpha=0.3)
axes[0].set_ylabel("test accuracy"); axes[0].legend()
plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/accuracy_vs_severity.png", dpi=150); plt.show()

## 7. Explanations: Grad-CAM (CNN) and attention rollout (ViT)
Both return a heatmap in [0, 1] at image resolution, plus the logits and predicted class.

In [ ]:
def normalize_map(m):
    B = m.shape[0]
    flat = m.reshape(B, -1)
    mn = flat.min(1, keepdim=True)[0]; mx = flat.max(1, keepdim=True)[0]
    return ((flat - mn) / (mx - mn + 1e-8)).reshape(m.shape)

def explain_cnn(model, x):
    """Grad-CAM on the last ResNet stage, for the predicted class."""
    model.eval()
    store = {}
    def fwd_hook(mod, inp, out):
        store["a"] = out
        out.register_hook(lambda g: store.__setitem__("g", g))
    h = model.m.layer4.register_forward_hook(fwd_hook)
    with torch.enable_grad():
        logits = model(x)
        pred = logits.argmax(1)
        score = logits.gather(1, pred[:, None]).sum()
        model.zero_grad(set_to_none=True)
        score.backward()
    h.remove()
    a, g = store["a"].detach(), store["g"].detach()
    w = g.mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * a).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
    return normalize_map(cam)[:, 0], logits.detach(), pred

def explain_vit(model, x):
    """Attention rollout (Abnar & Zuidema, 2020) for the CLS token."""
    model.eval()
    attn_store, hooks, restore = [], [], []
    for blk in model.m.blocks:
        if hasattr(blk.attn, "fused_attn"):
            restore.append((blk.attn, blk.attn.fused_attn)); blk.attn.fused_attn = False
        hooks.append(blk.attn.attn_drop.register_forward_hook(lambda mod, inp, out: attn_store.append(out.detach())))
    with torch.no_grad():
        logits = model(x)
    for h in hooks: h.remove()
    for mod, val in restore: mod.fused_attn = val
    if len(attn_store) != len(model.m.blocks):
        raise RuntimeError("Could not capture attention maps. Check your timm version (try: pip install timm==1.0.9).")
    B, N = x.shape[0], attn_store[0].shape[-1]
    eye = torch.eye(N, device=x.device)
    result = eye.unsqueeze(0).repeat(B, 1, 1)
    for a in attn_store:
        a = a.mean(1) + eye                      # average heads, add residual connection
        a = a / a.sum(-1, keepdim=True)
        result = a @ result
    mask = result[:, 0, 1:]                      # CLS token -> patch tokens
    side = int(mask.shape[-1] ** 0.5)
    mask = F.interpolate(mask.reshape(B, 1, side, side), size=x.shape[-2:], mode="bilinear", align_corners=False)
    return normalize_map(mask)[:, 0], logits, logits.argmax(1)

EXPLAIN = {"ResNet-18 (CNN)": explain_cnn, "DeiT-Tiny (ViT)": explain_vit}

In [ ]:
def overlay(ax, img, cam, title):
    ax.imshow(img); ax.imshow(cam, cmap="jet", alpha=0.45); ax.set_title(title, fontsize=8); ax.axis("off")

def explanation_figure(corruption="fog", severity=3, n_images=4):
    xs, ys = next(iter(test_loader))
    xs, ys = xs[:n_images].to(DEVICE), ys[:n_images].to(DEVICE)
    set_seed(); x_bad = CORRUPTIONS[corruption](xs, severity)
    out = {}
    for tag, inp in [("clean", xs), ("bad", x_bad)]:
        out[(tag, "cnn")] = explain_cnn(models["ResNet-18 (CNN)"], inp)
        out[(tag, "vit")] = explain_vit(models["DeiT-Tiny (ViT)"], inp)
    fig, axes = plt.subplots(n_images, 6, figsize=(15, 2.7 * n_images))
    for i in range(n_images):
        img_c = xs[i].permute(1, 2, 0).cpu().numpy(); img_b = x_bad[i].permute(1, 2, 0).cpu().numpy()
        axes[i, 0].imshow(img_c); axes[i, 0].set_title(f"clean (true class {ys[i].item()})", fontsize=8); axes[i, 0].axis("off")
        axes[i, 3].imshow(img_b); axes[i, 3].set_title(f"{corruption}, severity {severity}", fontsize=8); axes[i, 3].axis("off")
        for col, tag, kind, img in [(1, "clean", "cnn", img_c), (2, "clean", "vit", img_c), (4, "bad", "cnn", img_b), (5, "bad", "vit", img_b)]:
            cam, logits, pred = out[(tag, kind)]
            ok = "OK" if pred[i].item() == ys[i].item() else "WRONG"
            name = "CNN Grad-CAM" if kind == "cnn" else "ViT rollout"
            overlay(axes[i, col], img, cam[i].cpu().numpy(), f"{name} | pred {pred[i].item()} {ok}")
    plt.tight_layout(); plt.savefig(f"{CFG['out_dir']}/explanations_{corruption}.png", dpi=150); plt.show()

explanation_figure("fog", 3)
explanation_figure("blur", 3)

## 8. Faithfulness check (deletion test)
For each image: remove the top 20% most-highlighted pixels (replace with the mean colour) and measure how much the model's confidence in its original prediction drops. Compare with removing a *random* 20% region. If the explanation is faithful, removing the highlighted region should hurt more than removing a random one.

In [ ]:
def deletion_test(name, n_images=200, frac=0.2):
    model, explain = models[name], EXPLAIN[name]
    drops_expl, drops_rand, seen = [], [], 0
    for x, _ in test_loader:
        if seen >= n_images: break
        x = x.to(DEVICE)
        cam, logits, pred = explain(model, x)
        p0 = F.softmax(logits, 1).gather(1, pred[:, None])[:, 0]
        B = x.shape[0]; k = int(frac * cam[0].numel())
        flat = cam.reshape(B, -1)
        thr = flat.topk(k, dim=1).values[:, -1:]
        m_expl = (flat >= thr).reshape(B, 1, *x.shape[-2:]).float()
        rnd = torch.rand_like(flat)
        m_rand = (rnd >= rnd.topk(k, dim=1).values[:, -1:]).reshape(B, 1, *x.shape[-2:]).float()
        with torch.no_grad():
            p_e = F.softmax(model(x * (1 - m_expl) + model.mean * m_expl), 1).gather(1, pred[:, None])[:, 0]
            p_r = F.softmax(model(x * (1 - m_rand) + model.mean * m_rand), 1).gather(1, pred[:, None])[:, 0]
        drops_expl += (p0 - p_e).detach().cpu().tolist(); drops_rand += (p0 - p_r).detach().cpu().tolist()
        seen += B
    return dict(model=name, n_images=len(drops_expl), mean_drop_explanation=float(np.mean(drops_expl)), mean_drop_random=float(np.mean(drops_rand)))

set_seed()
faith = pd.DataFrame([deletion_test(n) for n in models])
faith.to_csv(f"{CFG['out_dir']}/faithfulness.csv", index=False)
print(faith.round(3))

## 9. Size and latency
Measured on the Colab GPU, so this is a **relative** comparison between the two models, not an edge-device result.

In [ ]:
@torch.no_grad()
def latency_ms_per_image(model, batch, runs=50):
    model.eval(); x = torch.rand(batch, 3, CFG["img_size"], CFG["img_size"], device=DEVICE)
    for _ in range(10): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(runs): model(x)
    if DEVICE == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - t) / runs / batch * 1000

lat = pd.DataFrame([dict(model=n,
                         params_millions=sum(p.numel() for p in m.parameters()) / 1e6,
                         ms_per_image_batch1=latency_ms_per_image(m, 1),
                         ms_per_image_batch32=latency_ms_per_image(m, 32),
                         device=(torch.cuda.get_device_name(0) if DEVICE == "cuda" else "cpu"))
                    for n, m in models.items()])
lat.to_csv(f"{CFG['out_dir']}/latency.csv", index=False)
print(lat.round(3))

## 10. Save a summary (copy these real numbers into your README)

In [ ]:
def to_md(d):
    cols = list(d.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in d.iterrows():
        lines.append("| " + " | ".join(f"{v:.3f}" if isinstance(v, float) else str(v) for v in r) + " |")
    return "\n".join(lines)

pivot = df.pivot_table(index=["corruption", "severity"], columns="model", values="accuracy").reset_index()
print("### Accuracy\n" + to_md(pivot))
print("\n### Faithfulness (deletion test)\n" + to_md(faith))
print("\n### Size and latency\n" + to_md(lat))

with open(f"{CFG['out_dir']}/summary.json", "w") as f:
    json.dump(dict(config=CFG, seed=SEED, accuracy=df.to_dict("records"),
                   faithfulness=faith.to_dict("records"), latency=lat.to_dict("records")), f, indent=2)
print("\nSaved everything to", CFG["out_dir"], "- download this folder and add it to your repo.")

## 11. Your analysis (write this in your own words)
Look at the plots and tables, then answer, in 5-8 sentences each, in the README:
1. Which model lost more accuracy under each corruption, and by how much?
2. In the explanation figures, how did each model's focus change from clean to degraded images? Did the highlights move away from the sign?
3. Did the deletion test suggest the explanations are faithful for both models?
4. What is the trade-off between accuracy, robustness and latency here?
5. What are the limits of this study, and what would you test next?